# Handel RL: Phase 13 analysis

Rebuilds every Phase 13 chart in `docs/rl/charts/` from the saved run data, and prints the
ablation table. Nothing here trains or plays games except where a cell says so, and nothing
calls a paid API.

**Inputs** (in `runs/`, which is git-ignored, so they exist only on the machine that made them):

| Charts | Input | Made by |
|---|---|---|
| learning curve, diagnostics, Elo of each stage | `runs/<run id>/eval.csv`, `train.csv`, `elo.csv` | `handel-train` |
| `trade_modes.png`, `stages_curve.png` | the saved `handel-rl-eval` files listed in `handel/rl/report_charts.py` | `handel-rl-eval --trade-mode ...` |
| `strategy_*.png` | `runs/phase13/strategy/league_4p_full_s0_best.json` | `python -m handel.rl.strategy` |
| `value_*.png` | `runs/phase13/value/league_4p_full_s0_best.json` | `python -m handel.rl.valuecheck` |
| `ablation_*.png` | `runs/abl_*_s0` and `_s1` (each with `ablation/*.json`) | `handel-train` with the ablation overrides, then `python -m handel.rl.ablation suite` |

A chart whose input is missing is skipped with a note. Run all cells with
`uv sync --all-groups` installed (the `rl` and `analysis` groups). Set `OUT` to another folder to
draw the charts without touching the tracked ones. The write-up is `docs/rl/RESULTS.md`.

In [ ]:
import json
import os
from pathlib import Path

from IPython.display import Image, Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "runs").exists() else Path.cwd().parent
os.chdir(ROOT)  # every input path is relative to the repository root
OUT = ROOT / "docs/rl/charts"
OUT.mkdir(parents=True, exist_ok=True)


def show(paths):
    for p in paths:
        if p is None:
            continue
        display(Image(str(p)))


print(f"repository: {ROOT}\ncharts go to: {OUT}")

## 1. Training through the stages

The learning curve, diagnostics and (for league runs) Elo chart of each stage's main run:
Phase 9 (fixed bots), Phase 10 (self-play league), Phase 11 (trade answers), Phase 12 (trade
proposals). Then all four curves on one axis, each stage starting from the one before.

In [ ]:
from handel.rl import curves, report_charts

STAGES = ["ppo_4p_vs_bots_long_s0", "league_4p_s0", "league_4p_respond_s0", "league_4p_full_s0"]
for run in STAGES:
    d = ROOT / "runs" / run
    if not d.exists():
        print(f"{run}: not on this machine, skipped")
        continue
    made = [curves.learning_curve(d, OUT / f"{run}_curve.png"), curves.diagnostics(d, OUT / f"{run}_diagnostics.png"),
            curves.elo_chart(d, OUT / f"{run}_elo.png")]
    print(run, "->", ", ".join(p.name for p in made if p))
show([OUT / "league_4p_full_s0_curve.png", OUT / "league_4p_full_s0_elo.png"])

In [ ]:
show([report_charts.stages_curve_chart(OUT / "stages_curve.png")])

## 2. What each part of trading is worth

The Phase 10, 11 and 12 agents in every trade mode, on the same 4,000 games against 3
`heuristic`. Learned answers are worth about 5 points, the heuristic bot's proposals about 12,
the agent's own proposals 17 more.

In [ ]:
rows = report_charts.trade_mode_table()
lines = ["| stage | trade mode | win rate | 95% interval | proposals / game |", "|---|---|---|---|---|"]
for r in rows:
    ci = r["ci"] or (None, None)
    lines.append(f"| {r['stage']} | `{r['mode']}` | {r['win_rate']:.1%} | "
                 f"[{ci[0]:.1%}, {ci[1]:.1%}] | {r['proposals_per_game']:.2f} |" if ci[0] is not None else
                 f"| {r['stage']} | `{r['mode']}` | {r['win_rate']:.1%} | - | {r['proposals_per_game']:.2f} |")
display(Markdown("\n".join(lines)))
show([report_charts.trade_modes_chart(OUT / "trade_modes.png")])

## 3. What strategy it learned

The Phase 12 agent and the heuristic bot in the same seat on the same seeds (4,000 games each,
against 3 `heuristic`): buying, building, jail and cash.

In [ ]:
show(report_charts.strategy_charts(OUT))

## 4. Does the value head know when it is winning?

Calibration against real outcomes, and the value estimate next to Monte Carlo win
probabilities (the Phase 4 judge) over a few games.

In [ ]:
doc = report_charts._load(report_charts.VALUE_FILE)
if doc:
    m = doc["mc_summary"]
    print(f"Monte Carlo positions: {m['positions']}; correlation with the value head: "
          f"as played {m['value_vs_played']['corr']:+.2f}, heuristic bots everywhere {m['value_vs_heuristic']['corr']:+.2f}")
show(report_charts.value_charts(OUT))

## 5. Ablations (B0 to B5, 2 training seeds each, 50M decisions)

Every arm minus the control of the same seed, on the same games, both seeds combined. The
first interval counts evaluation noise and the paired seed noise (what is left of the seed's
effect when arm and control share a seed); "full seed noise" is the stricter interval as if they
shared nothing; "eval only" leaves the seed out. "real" means the first interval excludes 0 and
both seeds agree; "robust" that the strict one excludes 0 too. The method is in the docstring of
`handel/rl/ablation_stats.py`.
This cell reads the 12 suites (about 10 seconds) and writes `runs/phase13/ablation_summary.json`.

In [ ]:
from handel.rl import ablation_charts, ablation_stats

summary = ablation_stats.analyse(ROOT / "runs")
Path(ablation_stats.OUT).parent.mkdir(parents=True, exist_ok=True)
Path(ablation_stats.OUT).write_text(json.dumps(summary, indent=1), encoding="utf-8")
print(ablation_stats.format_table(summary))

In [ ]:
show(ablation_charts.all_charts(OUT, ablation_stats.OUT))

## 6. Seed noise

How much the training seed moves a result (standard deviation, points of win rate). *Whole run*:
two runs of the same arm with different seeds (seed 0 minus seed 1 is listed for every arm).
*Paired*: what is left of it in "arm minus control of the same seed", because runs with the same
seed share their random numbers. Two single runs with different seeds need to differ by well
over 1.4 times the whole-run figure before the difference means anything.

In [ ]:
for metric, label, unit in ablation_stats.METRICS:
    n = summary["seed_noise"].get(metric)
    if not n or unit != "share":
        continue
    gaps = ", ".join(f"{a.split('_')[1].upper()} {g * 100:+.1f}" for a, g in n["run_gaps"].items())
    print(f"{label:<42} whole run sd {(n['full_sd'] or 0) * 100:.1f}, paired sd {(n['paired_sd'] or 0) * 100:.1f} points"
          f"   (seed 0 minus seed 1: {gaps})")